# SYK on a Quantum Computer: Magic and Entanglement of $q=2$ vs $q=4$

## Motivation

The Sachdev–Ye–Kitaev (SYK) model is a touchstone of quantum chaos: all-to-all
random couplings between Majorana fermions, maximally chaotic at low temperature,
and holographically dual to a black hole. Through the **Jordan–Wigner
transformation** it is also an explicit **qubit Hamiltonian** — something you can
put on a quantum computer and run.

That makes SYK an ideal laboratory for the central question of this book: *what
makes a quantum system classically hard?* We will build qubit-SYK for two cases,

* $q=4$ — the genuine four-body model: **chaotic and non-Gaussian**,
* $q=2$ — a two-body model: **free fermions (Gaussian)**,

and measure two resources on the resulting states: the **half-cut entanglement
entropy** and the **stabilizer Rényi entropy** (the *magic*, i.e. distance from
the stabilizer states). The punchline, which we will reach numerically, is that
**magic alone cannot tell the hard model from the easy one** — the free $q=2$
model carries just as much magic as the chaotic $q=4$ model, yet is classically
simulable. Understanding *why* is the point of the notebook.

## Problem

1. Map the Majorana SYK model to qubits with the Jordan–Wigner transformation,
   and build the Hamiltonian for $q=2$ and $q=4$.
2. Measure the half-cut entanglement and the magic $M_2$ of the mid-spectrum
   eigenstate, for $q=2$, $q=4$, and Haar-random states.
3. Watch both quantities **grow in time** when the model is run as a circuit from
   the register state $|0\cdots0\rangle$.
4. Build an **efficient Pauli-string simulator** that reaches larger systems, and
   verify it against the brute-force construction.
5. Explain the result: $q=2$ is a **free-fermion (Gaussian)** model, classically
   simulable by matchgate methods — a *different* island of easiness from the
   stabilizer formalism that magic measures.

## 1. Jordan–Wigner: from Majoranas to qubits

A set of $N_M=2L$ Majorana operators $\chi_a$ obeys $\{\chi_a,\chi_b\}=2\delta_{ab}$
(they square to the identity and anticommute). The Jordan–Wigner transformation
represents them on $L$ qubits as Pauli strings:
$$
\chi_{2k} = Z_0Z_1\cdots Z_{k-1}\,X_k,\qquad
\chi_{2k+1} = Z_0Z_1\cdots Z_{k-1}\,Y_k,\qquad k=0,\dots,L-1.
$$
The leading string of $Z$'s (the "Jordan–Wigner string") is what enforces the
fermionic anticommutation across different sites. Let us build these as explicit
matrices and check the algebra.

In [ ]:
import numpy as np
from itertools import combinations
from functools import reduce
import matplotlib.pyplot as plt

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def kron(ops):
    return reduce(np.kron, ops)

def dense_majoranas(L):
    # Convention: qubit k is bit k (qubit 0 = least significant = rightmost kron
    # factor). chi_{2k} = Z_0..Z_{k-1} X_k  ->  [I]*(L-1-k) + [X] + [Z]*k
    chis = []
    for k in range(L):
        chis.append(kron([I2]*(L-1-k) + [X] + [Z]*k))
        chis.append(kron([I2]*(L-1-k) + [Y] + [Z]*k))
    return chis

# check {chi_a, chi_b} = 2 delta_ab on L=3 qubits
L = 3
chi = dense_majoranas(L)
Iden = np.eye(2**L)
ok = all(np.allclose(chi[a] @ chi[b] + chi[b] @ chi[a],
                     2*Iden if a == b else 0*Iden)
         for a in range(2*L) for b in range(2*L))
print("Majorana algebra {chi_a, chi_b} = 2 delta_ab :", ok)

## 2. The SYK Hamiltonian

The Majorana SYK Hamiltonian couples the fermions with real Gaussian random
couplings:
$$
q=4:\quad H=\!\!\sum_{i<j<k<l}\!\! J_{ijkl}\,\chi_i\chi_j\chi_k\chi_l,
\qquad
q=2:\quad H=\sum_{i<j} J_{ij}\,(i\,\chi_i\chi_j).
$$
For $q=4$ the product of four distinct Majoranas is already Hermitian; for $q=2$
the bilinear $\chi_i\chi_j$ is *anti*-Hermitian, so we include an explicit factor
$i$. The variances ($3!\,J^2/N_M^3$ for $q=4$, $J^2/N_M$ for $q=2$) keep the
energy extensive. Diagonalising, the spectrum is symmetric about zero.

In [ ]:
def dense_syk(L, q, rng, J=1.0):
    chi = dense_majoranas(L)
    Nm = 2*L
    H = np.zeros((2**L, 2**L), dtype=complex)
    if q == 4:
        sd = np.sqrt(6.0 * J**2 / Nm**3)
        for i, j, k, l in combinations(range(Nm), 4):
            H += rng.normal(0, sd) * (chi[i] @ chi[j] @ chi[k] @ chi[l])
    elif q == 2:
        sd = np.sqrt(J**2 / Nm)
        for i, j in combinations(range(Nm), 2):
            H += rng.normal(0, sd) * (1j * chi[i] @ chi[j])
    return 0.5 * (H + H.conj().T)   # discard tiny numerical anti-Hermitian part

H4 = dense_syk(4, 4, np.random.default_rng(0))
print("q=4, L=4 Hermitian:", np.allclose(H4, H4.conj().T))
E = np.linalg.eigvalsh(H4)
print(f"spectrum symmetric about 0?  min={E.min():.3f}  max={E.max():.3f}")

## 3. The two resources: entanglement and magic

**Entanglement.** Split the $L$ qubits in half and compute the von Neumann
entropy of the reduced state — the standard half-cut entanglement entropy.

**Magic (stabilizer Rényi entropy).** The magic measures how far a state is from
the *stabilizer* states (those reachable by Clifford gates). For a pure state,
$$
M_2(\psi) = -\log_2\!\Big[\tfrac{1}{D}\sum_{P}\langle\psi|P|\psi\rangle^4\Big],
$$
where the sum runs over all $4^L$ Pauli strings $P$ and $D=2^L$. It is $0$ for
stabilizer states and approaches $\approx L-\log_2 3$ for Haar-random states.

A naive evaluation costs $4^L$; we use a fast $O(L\,4^L)$ Walsh–Hadamard method
(grouping Paulis by their $X$-part), which we will explain when we need it.

In [ ]:
def half_entanglement(psi, L):
    a = L // 2
    s = np.linalg.svd(psi.reshape(2**a, 2**(L-a)), compute_uv=False)
    p = s**2; p = p[p > 1e-13]
    return float(-np.sum(p * np.log(p)))

def _fwht_rows(M, N):
    # batched Walsh-Hadamard along axis 1 (radix-2 butterfly); output order is
    # irrelevant since we sum |.|^4 over all outputs.
    rows, D = M.shape
    T, h = M, 1
    while h < D:
        T = T.reshape(rows, D//(2*h), 2*h)
        a, b = T[:, :, :h], T[:, :, h:]
        T = np.concatenate([a+b, a-b], axis=2).reshape(rows, D)
        h *= 2
    return T

def magic_M2(psi, L):
    # Group the 4^L Paulis by X-mask x: the 2^L expectations sharing x are the
    # Walsh-Hadamard transform of C_x[j] = conj(psi_j) psi_{j xor x}. Build all
    # C_x as one D x D matrix, transform once, sum the 4th powers.
    psi = psi / np.linalg.norm(psi)
    D = 1 << L
    j = np.arange(D)
    Mx = np.conj(psi)[None, :] * psi[j[:, None] ^ j[None, :]]
    F = _fwht_rows(Mx, L)
    return float(-np.log2(np.sum(np.abs(F)**4) / D))

def haar_state(L, rng):
    v = rng.normal(size=1 << L) + 1j*rng.normal(size=1 << L)
    return v/np.linalg.norm(v)

# sanity: a stabilizer state has magic 0; a Haar state ~ L - log2(3)
zero = np.zeros(2**4); zero[0] = 1.0
print("M2(|0000>)      =", round(magic_M2(zero, 4), 6), " (stabilizer -> 0)")
print("M2(Haar, L=4)   =", round(magic_M2(haar_state(4, np.random.default_rng(1)), 4), 3),
      " (~ L - log2 3 =", round(4 - np.log2(3), 3), ")")

## 4. Eigenstate comparison: $q=2$ vs $q=4$ vs Haar

We take the **mid-spectrum eigenstate** (the centre of the spectrum, where the
model is maximally chaotic) and average its entanglement and magic over disorder
realisations. Watch the magic column: $q=2$ (free fermions) is **just as
magical** as $q=4$ (chaotic).

In [ ]:
def eigenstate_S_M(L, q, n_real):
    S, M = [], []
    for r in range(n_real):
        H = dense_syk(L, q, np.random.default_rng(100 + r))
        _, V = np.linalg.eigh(H)
        psi = V[:, 2**L // 2]           # middle eigenstate
        S.append(half_entanglement(psi, L)); M.append(magic_M2(psi, L))
    return np.mean(S), np.mean(M)

print(f"{'L':>3} {'q':>3} | {'S (entanglement)':>17} | {'M2 (magic)':>11}")
for L in (4, 6):
    for q in (2, 4):
        s, m = eigenstate_S_M(L, q, 8)
        print(f"{L:>3} {q:>3} | {s:>17.3f} | {m:>11.3f}")
    # Haar reference at this L
    hs = [haar_state(L, np.random.default_rng(500+r)) for r in range(8)]
    print(f"{L:>3} {'Haar':>3} | {np.mean([half_entanglement(p,L) for p in hs]):>17.3f} | "
          f"{np.mean([magic_M2(p,L) for p in hs]):>11.3f}")

Two lessons already: $q=4$ entanglement sits near the Page value (maximal), while
$q=2$ is a little lower; but the **magic is essentially the same for $q=2$ and
$q=4$**. If magic controlled hardness, the free model would look just as hard as
the chaotic one — which cannot be right, since free fermions are classically
easy. We resolve this in Section 7.

## 5. An efficient simulator: SYK as Pauli strings

To reach larger $L$ we avoid dense matrices. The key fact is that **every product
of Majoranas is a single Pauli string** — a tensor product of $I,X,Y,Z$. We store
a Pauli string as two binary numbers plus a phase:

> A Pauli string is "$X$ on some qubits, $Z$ on others." Record which qubits get
> an $X$ in a bitmask $x$, and which get a $Z$ in a bitmask $z$ (a qubit with both
> is a $Y$, up to a factor $i$). The whole operator is $i^p\,X^x Z^z$.

Why this is powerful: because $X^2=Z^2=I$, multiplying two Pauli strings just
**XORs the masks**, $x=x_1\oplus x_2$, $z=z_1\oplus z_2$; the only real work is a
sign. Sliding $Z^{z_1}$ past $X^{x_2}$ flips a sign once for every qubit carrying
both, i.e. a factor $(-1)^{\mathrm{popcount}(z_1\,\&\,x_2)}$, where *popcount*
counts the 1-bits. (This binary picture is the "symplectic" or "tableau"
representation; the name is just bookkeeping.)

Applying $i^pX^xZ^z$ to a basis state $|j\rangle$ is likewise cheap:
$$
i^pX^xZ^z\,|j\rangle = i^p\,(-1)^{\mathrm{popcount}(z\,\&\,j)}\,|j\oplus x\rangle,
$$
— the $Z$-mask contributes a $-1$ for each qubit where both $z$ and $j$ are $1$,
and the $X$-mask flips the qubits in $x$. So one Pauli string acts on a whole
statevector in $O(2^L)$: permute the amplitudes by $j\mapsto j\oplus x$ and flip
some signs. Below, `apply_H` is exactly this formula, vectorised.

In [ ]:
# 16-bit popcount table. MUST be signed: the sign 1 - 2*(popcount & 1) would
# underflow to 255 in unsigned arithmetic instead of giving -1.
_POP = np.array([bin(i).count("1") for i in range(1 << 16)], dtype=np.int64)
_PH = np.array([1, 1j, -1, -1j])   # i^p

def ps_majoranas(L):
    # chi_{2k} = Z..Z X_k : x = bit k, z = low k bits, p=0
    # chi_{2k+1}= Z..Z Y_k : Y adds a Z on qubit k -> z gets bit k, p=1
    ops = []
    for k in range(L):
        xk, zstr = 1 << k, (1 << k) - 1
        ops.append((xk, zstr, 0)); ops.append((xk, zstr | xk, 1))
    return ops

def pmul(A, B):
    (x1, z1, p1), (x2, z2, p2) = A, B
    return (x1 ^ x2, z1 ^ z2, (p1 + p2 + 2*bin(z1 & x2).count("1")) & 3)

def ps_syk_terms(L, q, rng, J=1.0):
    chi, Nm, terms = ps_majoranas(L), 2*L, []
    if q == 4:
        sd = np.sqrt(6.0*J**2/Nm**3)
        for idx in combinations(range(Nm), 4):
            P = chi[idx[0]]
            for a in idx[1:]: P = pmul(P, chi[a])
            terms.append((float(rng.normal(0, sd)), *P))
    else:
        sd = np.sqrt(J**2/Nm)
        for i, j in combinations(range(Nm), 2):
            x, z, p = pmul(chi[i], chi[j])
            terms.append((float(rng.normal(0, sd)), x, z, (p+1) & 3))
    return terms

def apply_H(psi, terms, L):
    j = np.arange(1 << L, dtype=np.int64)
    out = np.zeros(1 << L, dtype=complex)
    for c, x, z, p in terms:
        jx = j ^ x
        sign = 1 - 2*(_POP[z & jx] & 1)          # (-1)^popcount(z & (j^x))
        out += (c*_PH[p]) * sign * psi[jx]
    return out

# VERIFY the Pauli-string H equals the dense H for identical couplings
L = 4
for q in (2, 4):
    terms = ps_syk_terms(L, q, np.random.default_rng(9))
    # rebuild the dense matrix from the same terms
    Hd = np.zeros((1 << L, 1 << L), complex); jj = np.arange(1 << L)
    for c, x, z, p in terms:
        Hd[jj ^ x, jj] += c*_PH[p]*(1 - 2*(_POP[z & jj] & 1))
    v = haar_state(L, np.random.default_rng(3))
    print(f"q={q}: apply_H matches dense matrix:", np.allclose(apply_H(v, terms, L), Hd @ v))

## 6. Scaling the magic with the efficient simulator

With the Pauli-string Hamiltonian we compute the mid-spectrum eigenstate magic
out to larger $L$ (here we still diagonalise a rebuilt dense matrix for the
eigenstate, but the same machinery evolves states without any matrix at all — see
the next section). The magic grows roughly linearly in $L$ for **both** $q=2$ and
$q=4$: it is *extensive* in each case, so no threshold in magic separates them.

In [ ]:
def ps_dense(terms, L):
    Hd = np.zeros((1 << L, 1 << L), complex); j = np.arange(1 << L)
    for c, x, z, p in terms:
        Hd[j ^ x, j] += c*_PH[p]*(1 - 2*(_POP[z & j] & 1))
    return Hd

Ls = [4, 6, 8]
res = {2: [], 4: []}
for L in Ls:
    for q in (2, 4):
        ms = []
        for r in range(6):
            H = ps_dense(ps_syk_terms(L, q, np.random.default_rng(200+r)), L)
            _, V = np.linalg.eigh(H)
            ms.append(magic_M2(V[:, 2**L // 2], L))
        res[q].append(np.mean(ms))
    print(f"L={L}: M2(q=2)={res[2][-1]:.3f}  M2(q=4)={res[4][-1]:.3f}")

plt.figure(figsize=(5, 3.5))
plt.plot(Ls, res[4], "o-", color="#c1121f", label="q=4 (chaotic)")
plt.plot(Ls, res[2], "s-", color="#4361ee", label="q=2 (free fermions)")
plt.xlabel("qubits L"); plt.ylabel("magic $M_2$"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Running SYK on a quantum computer: dynamics from $|0\cdots0\rangle$

Now the "quantum computer" reading: start from the register $|0\cdots0\rangle$
(a stabilizer state, so $S=0$ and $M_2=0$) and evolve $e^{-iHt}|0\cdots0\rangle$.
We propagate with a short Lanczos (Krylov) matrix-exponential — only
matrix–vector products, no diagonalisation — so this scales to large $L$. Both
entanglement and magic **rise from zero and saturate**, and the $q=2$ magic
plateau again lands right beside $q=4$.

In [ ]:
def lanczos_expm(psi, terms, L, t, m=30):
    beta0 = np.linalg.norm(psi); V = np.zeros((m, len(psi)), complex)
    al = np.zeros(m); be = np.zeros(m); V[0] = psi/beta0
    w = apply_H(V[0], terms, L); al[0] = np.vdot(V[0], w).real; w -= al[0]*V[0]
    for k in range(1, m):
        be[k] = np.linalg.norm(w)
        if be[k] < 1e-12: m = k; break
        V[k] = w/be[k]; w = apply_H(V[k], terms, L)
        al[k] = np.vdot(V[k], w).real; w = w - al[k]*V[k] - be[k]*V[k-1]
    T = np.diag(al[:m]) + np.diag(be[1:m], 1) + np.diag(be[1:m], -1)
    ev, evec = np.linalg.eigh(T)
    return beta0 * ((evec @ (np.exp(-1j*t*ev) * evec[0].conj())) @ V[:m])

L = 8
times = np.concatenate([np.arange(0, 2.01, 0.2), [3., 4., 6., 8.]])
curves = {}
for q in (2, 4):
    Sacc = np.zeros(len(times)); Macc = np.zeros(len(times)); nr = 6
    for r in range(nr):
        terms = ps_syk_terms(L, q, np.random.default_rng(1234+r))
        psi = np.zeros(1 << L, complex); psi[0] = 1.0; tp = 0.0
        for it, t in enumerate(times):
            if t > tp: psi = lanczos_expm(psi, terms, L, t-tp); tp = t
            Sacc[it] += half_entanglement(psi, L)/nr; Macc[it] += magic_M2(psi, L)/nr
    curves[q] = (Sacc, Macc)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
for q, c in ((4, "#c1121f"), (2, "#4361ee")):
    lab = "q=4 (chaotic)" if q == 4 else "q=2 (free fermions)"
    a1.plot(times, curves[q][0], color=c, label=lab)
    a2.plot(times, curves[q][1], color=c, label=lab)
a1.set_xlabel("time t"); a1.set_ylabel("entanglement S(t)"); a1.legend()
a2.set_xlabel("time t"); a2.set_ylabel("magic $M_2$(t)"); a2.legend()
plt.tight_layout(); plt.show()

## 8. Why is $q=2$ easy? The two free islands

We have a paradox to resolve: $q=2$ has as much magic and nearly as much
entanglement as $q=4$, yet $q=2$ is classically **easy**. The resolution is that
magic measures distance from **one** island of classically-simulable states — the
**stabilizer** states, reachable by Clifford gates. There is a **second**,
independent island: **free-fermion (Gaussian) states**, reachable by *matchgate*
circuits and simulated with completely different machinery (the $O(L^2)$
covariance matrix, via Wick's theorem).

The $q=2$ model is bilinear in Majoranas, so its dynamics are Gaussian: its states
are fully determined by their two-point functions, and every higher correlator
factorises by Wick's theorem. That is exactly what makes them simulable — and it
has nothing to do with stabilizers, so these states can be (and are) highly
magical. Let us *see* the Gaussianity: for a $q=2$ eigenstate the four-point
Majorana correlator equals the Wick (Pfaffian) sum of two-point functions, while
for $q=4$ it does not.

In [ ]:
def two_point(psi, chi, a, b):
    return np.vdot(psi, chi[a] @ (chi[b] @ psi))

def four_point(psi, chi, a, b, c, d):
    return np.vdot(psi, chi[a] @ (chi[b] @ (chi[c] @ (chi[d] @ psi))))

def wick(psi, chi, a, b, c, d):
    G = lambda i, j: two_point(psi, chi, i, j)
    return G(a, b)*G(c, d) - G(a, c)*G(b, d) + G(a, d)*G(b, c)

L = 4; chi = dense_majoranas(L)
for q in (2, 4):
    H = dense_syk(L, q, np.random.default_rng(0)); _, V = np.linalg.eigh(H)
    psi = V[:, 2**L // 2]
    a, b, c, d = 0, 1, 2, 3
    exact = four_point(psi, chi, a, b, c, d)
    wk = wick(psi, chi, a, b, c, d)
    print(f"q={q}:  <chi chi chi chi> = {exact:+.3f}   Wick sum = {wk:+.3f}   "
          f"Gaussian (match)? {np.isclose(exact, wk, atol=1e-6)}")

For $q=2$ the four-point function equals its Wick decomposition — the state is
**Gaussian** — while for $q=4$ it does not. Gaussianity is the operational reason
$q=2$ is classically simulable, entirely separately from its magic.

## Summary

* Jordan–Wigner turns Majorana SYK into an explicit qubit Hamiltonian; every
  Majorana product is a single Pauli string, which gives an $O(2^L)$-per-term
  simulator with no dense matrix.
* Evolving $|0\cdots0\rangle$ under SYK generates **both** entanglement and magic,
  which saturate at the mid-spectrum eigenstate values.
* **Magic does not certify hardness.** The free-fermion $q=2$ model is as magical
  as the chaotic $q=4$ model, yet is classically easy — because it lives on the
  *Gaussian* island (matchgate-simulable), a different resource from
  stabilizerness. A model is classically hard only when it is off **both** free
  islands: non-stabilizer **and** non-Gaussian. That is $q=4$.